# IMPORTS

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract_all, regexp_replace,col, lit, size,  explode,split, element_at,col,udf,length,make_date,expr,weekofyear,collect_list,explode,max,min
from pyspark.sql.types import StringType
import json
import pandas as pd
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data read

In [0]:
prices = spark.read.format('delta').load('/Volumes/market-mood/processed/prices_clean') 
spotify = spark.read.format('delta').load('/Volumes/market-mood/processed/spotify_aggregated')
tweets = spark.read.format('delta').load('/Volumes/market-mood/processed/sentiment_scores')

In [0]:
prices.printSchema()
spotify.printSchema()
tweets.printSchema()

# Data join

In [0]:
years = spotify.select('year').distinct().collect()

In [0]:
years =[y['year'] for y in years]

In [0]:
dates = pd.DataFrame(columns=['year','date'])
for year in years:
    for month in range(1,13):
        for day in range(1,32):
            try:
                dates = pd.concat([dates,pd.DataFrame({'year':year,'date':datetime.date(2022,month,day)},index=[0])])
            except:
                pass
dates = spark.createDataFrame(dates)

dates = dates.withColumn('week',weekofyear(col('date')))


In [0]:
dates = dates.groupBy('year','week').agg(collect_list('date').alias('dayList'))#collect list hace una lista de los elementos del groupby, en este caso, las fechas agrupadas por semana y año

In [0]:
spotify = spotify.join(dates,on=['year','week'])

In [0]:
spotify.printSchema()

In [0]:
spotify = spotify.withColumn('date', explode(spotify['dayList']))

In [0]:
spotify_dates = [x['date'] for x in spotify.select('date').collect()]
tweets_dates = [x['date'] for x in tweets.select('date').collect()]
print(prices.filter(col('date').isin(spotify_dates)).count())
print(prices.filter(col('date').isin(tweets_dates)).count())


In [0]:
print(spotify.filter(col('date').isin(tweets_dates)).count())

In [0]:
prices_dates = [x['date'] for x in prices.select('date').collect()]

In [0]:
spotify = spotify.filter(col('date').isin(prices_dates))
tweets = tweets.filter(col('date').isin(prices_dates))


In [0]:
spotify.select(max('date')).show()
spotify.select(min('date')).show()
tweets.select(max('date')).show()
tweets.select(min('date')).show()